In [146]:
import pandas as pd
import json
import re
from datetime import datetime
from fuzzywuzzy import fuzz # python library that helps compare similarities between two texts
from fuzzywuzzy import process
import warnings
warnings.filterwarnings('ignore')

# Load both CSVs
print("Loading CSVs...")
batch1_df = pd.read_csv(r"/Users/grigoriostsakalis/.cache/kagglehub/datasets/osamahosamabdellatif/high-quality-invoice-images-for-ocr/versions/3/batch_1/batch_1/batch1_1.csv")
output_df = pd.read_csv(r"/Users/grigoriostsakalis/Desktop/UniAI/makeathon-2026-WHOAMI-inform/ocr/.data/working/output.csv")

print(f"batch1_1.csv: {len(batch1_df)} rows")
print(f"output.csv: {len(output_df)} rows")
print("\nReady for validation pipeline.")

Loading CSVs...
batch1_1.csv: 499 rows
output.csv: 50 rows

Ready for validation pipeline.


# PARAMETERS

In [147]:
FUZZY_MATCH_THRESHOLD = 90 
STRICT_NUMERIC_TOLERANCE = 0.1

In [148]:
# ============================================================================
# HELPER FUNCTIONS FOR PARSING AND COMPARISON
# ============================================================================

def parse_ocred_text(ocred_text):
    """
    Extract key fields from OCRed Text ground truth.
    Returns a dictionary with extracted values.
    """
    extracted = {
        'Seller Name': None,
        'Client Name': None,
        'Seller Tax ID': None,
        'Client Tax ID': None,
        'Invoice Number': None,
        'Invoice Date': None,
        'Net Worth': None,
        'VAT': None,
        'Gross Worth': None
    }
    
    # Tax ID pattern (XXX-XX-XXXX format)
    tax_id_pattern = r'\d{3}-\d{2}-\d{4}'
    
    # Invoice number pattern (8 digits)
    invoice_num_pattern = r'Invoice\s+(?:no|number):\s*(\d+)'
    
    # Date patterns (MM/DD/YYYY or YYYY-MM-DD or variations)
    date_patterns = [
        r'Date\s+of\s+issue:\s*(\d{1,2}/\d{1,2}/\d{4})',
        r'Date\s+of\s+issue:\s*(\d{4}-\d{1,2}-\d{1,2})',
    ]
    
    # Money pattern (dollar amounts)
    money_pattern = r'[\$\s]*(\d+\.?\d*)'
    
    lines = ocred_text.split('\n')
    
    # Extract invoice number
    for line in lines:
        match = re.search(invoice_num_pattern, line, re.IGNORECASE)
        if match:
            extracted['Invoice Number'] = match.group(1)
            break
    
    # Extract date
    for line in lines:
        for pattern in date_patterns:
            match = re.search(pattern, line, re.IGNORECASE)
            if match:
                extracted['Invoice Date'] = match.group(1)
                break
        if extracted['Invoice Date']:
            break
    
    # Extract tax IDs (looking for two of them - seller and client)
    tax_ids = re.findall(tax_id_pattern, ocred_text)
    if len(tax_ids) >= 1:
        extracted['Seller Tax ID'] = tax_ids[0]
    if len(tax_ids) >= 2:
        extracted['Client Tax ID'] = tax_ids[1]
    
    # Extract seller and client names (usually appear before Tax Id labels)
    seller_match = re.search(r'Seller:\s*([^\n]+)', ocred_text, re.IGNORECASE)
    if seller_match:
        extracted['Seller Name'] = seller_match.group(1).strip()
    
    client_match = re.search(r'Client:\s*([^\n]+)', ocred_text, re.IGNORECASE)
    if client_match:
        extracted['Client Name'] = client_match.group(1).strip()
    
    # Extract monetary values from Total line
    def clean_num(val):
        """Clean numeric value: remove spaces, convert comma to dot, convert to float"""
        if not val: return None
        val = re.sub(r'[$€£\s]', '', val).replace(',', '.')
        try:    return float(val)
        except: return None
    
    def extract_three_numbers_after_total(text):
        """
        Extracts the first 3 numeric values after "Total" keyword.
        Handles missing/inconsistent currency symbols.
        Returns (net_worth, vat, gross_worth) as floats or (None, None, None).
        """
        # Find Total line
        total_match = re.search(r'Total\s+(.*?)(?:\n|$)', text, re.IGNORECASE)
        if not total_match:
            return None, None, None
        
        total_line = total_match.group(1)
        
        # Extract all numeric sequences (including spaces and commas)
        numbers = re.findall(r'[\d\s,]+(?:\.\d+)?', total_line)
        
        if len(numbers) < 3:
            return None, None, None
        
        # Take first 3 numbers
        net = clean_num(numbers[0])
        vat = clean_num(numbers[1])
        gross = clean_num(numbers[2])
        
        return net, vat, gross
    
    # Try to extract monetary values
    net, vat, gross = extract_three_numbers_after_total(ocred_text)
    
    extracted['Net Worth'] = net
    extracted['VAT'] = vat
    extracted['Gross Worth'] = gross
    
    if net is None:
        print(f"⚠️  Could not extract monetary values from Invoice Number: {extracted['Invoice Number']}")
    
    return extracted


def extract_date_components(date_str):
    """
    Extract date components (year, month, day) from various date formats.
    Returns a tuple of (year, month, day) as integers, or None if parsing fails.
    Intelligently handles ambiguous formats like X/Y/YYYY by checking if values > 12.
    """
    if not date_str:
        return None
    
    date_str = str(date_str).strip()
    
    # First, try unambiguous formats
    unambiguous_formats = [
        '%Y-%m-%d',      # YYYY-MM-DD (unambiguous)
        '%Y/%m/%d',      # YYYY/MM/DD (unambiguous)
    ]
    
    for date_format in unambiguous_formats:
        try:
            dt = datetime.strptime(date_str, date_format)
            return (dt.year, dt.month, dt.day)
        except:
            continue
    
    # For ambiguous formats (X/Y/YYYY or X-Y-YYYY), try to disambiguate
    ambiguous_patterns = [
        (r'(\d{1,2})[/\-](\d{1,2})[/\-](\d{4})', ['%m/%d/%Y', '%d/%m/%Y']),
    ]
    
    for pattern, formats in ambiguous_patterns:
        match = re.match(pattern, date_str)
        if match:
            first, second, year = match.groups()
            first, second = int(first), int(second)
            
            # If first value > 12, it must be day (DD/MM/YYYY format)
            if first > 12:
                try:
                    dt = datetime.strptime(date_str, '%d/%m/%Y' if '/' in date_str else '%d-%m-%Y')
                    return (dt.year, dt.month, dt.day)
                except:
                    pass
            # If second value > 12, it must be day (MM/DD/YYYY format)
            elif second > 12:
                try:
                    dt = datetime.strptime(date_str, '%m/%d/%Y' if '/' in date_str else '%m-%d-%Y')
                    return (dt.year, dt.month, dt.day)
                except:
                    pass
            # Both <= 12: ambiguous, try MM/DD/YYYY first (US format)
            else:
                for date_format in formats:
                    try:
                        dt = datetime.strptime(date_str, date_format)
                        return (dt.year, dt.month, dt.day)
                    except:
                        continue
    
    # Try other formats as fallback
    fallback_formats = [
        '%m-%d-%Y',      # MM-DD-YYYY
        '%d-%m-%Y',      # DD-MM-YYYY
    ]
    
    for date_format in fallback_formats:
        try:
            dt = datetime.strptime(date_str, date_format)
            return (dt.year, dt.month, dt.day)
        except:
            continue
    
    return None


def fuzzy_compare(str1, str2, threshold=FUZZY_MATCH_THRESHOLD):
    """
    Compare two strings with fuzzy matching.
    Returns: (similarity_score, is_match)
    """
    if not str1 or not str2:
        return (0, False)
    
    str1 = str(str1).strip().lower()
    str2 = str(str2).strip().lower()
    
    if str1 == str2:
        return (100, True)
    
    similarity = fuzz.token_set_ratio(str1, str2)
    is_match = similarity >= threshold
    
    return (similarity, is_match)


def strict_compare(val1, val2, tolerance=STRICT_NUMERIC_TOLERANCE):
    """
    Compare numbers strictly.
    Returns: (match_bool, difference)
    """
    if val1 is None or val2 is None:
        return (False, None)
    
    try:
        v1 = float(val1)
        v2 = float(val2)
        diff = abs(v1 - v2)

        # Allow small tolerance for floating point
        is_match = (diff <= tolerance)

        return (is_match, diff)

    except:
        return (False, None)

In [149]:
# examples
idx = 1

# print("\nExample Ground Truth json data for validation:\n", batch1_df['Json Data'].iloc[idx])
# print("="*80)
# print("\nExample Ocred text data for validation:\n", batch1_df['OCRed Text'].iloc[idx])

# sample= batch1_df['OCRed Text'].iloc[idx]

# parse_ocred_text(sample)


In [150]:
# ============================================================================
# MAIN VALIDATION FUNCTION
# ============================================================================

# See again domain Seller. Where seller name or seller
# Also domain Invoice Number. for some reason it finds mismatch when they are actually the same.
FUZZY_MATCH_THRESHOLD = 90 
STRICT_NUMERIC_TOLERANCE = 0.1
def validate_ocr_results(output_df, domains=None, tolerance=STRICT_NUMERIC_TOLERANCE, fuzzy_threshold=FUZZY_MATCH_THRESHOLD):
    """
    Validate OCR results against ground truth.
    
    Parameters:
    -----------
    domains : str or list of str, optional
        Specific domain(s) to validate. If None, validates all available domains.
        Available domains: 'Seller Name', 'Client Name', 'Seller Tax ID', 
                          'Client Tax ID', 'Invoice Number', 'Invoice Date',
                          'Net Worth', 'VAT', 'Gross Worth'
    
    Returns:
    --------
    validation_results : dict
        Comprehensive validation report with accuracy scores per domain
    """
    
    if domains is None:
        domains = ['Seller Name', 'Client Name', 'Seller Tax ID', 'Client Tax ID',
                   'Invoice Number', 'Invoice Date', 'Net Worth', 'VAT', 'Gross Worth']
    elif isinstance(domains, str):
        domains = [domains]
    
    # Initialize results structure
    results = {
        'domains_validated': domains,
        'total_files': len(output_df),
        'files_matched': 0,
        'files_missing': [],
        'domain_results': {domain: {
            'matches': 0,
            'mismatches': 0,
            'missing_in_ground_truth': 0,
            'missing_in_output': 0,
            'accuracy': 0.0,
            'details': []
        } for domain in domains},
        'summary': {}
    }
    
    # Create filename index for batch1_df for faster lookup
    batch1_index = {}
    for idx, row in batch1_df.iterrows():
        filename = row['File Name'].split('/')[-1]  # Get just filename
        batch1_index[filename] = idx
    
    # Validate each file in output.csv
    for out_idx, output_row in output_df.iterrows():
        filename = output_row['filename']
        
        # Find corresponding ground truth
        if filename not in batch1_index:
            results['files_missing'].append(filename)
            continue
        
        results['files_matched'] += 1
        batch1_idx = batch1_index[filename]
        batch1_row = batch1_df.iloc[batch1_idx]
        
        # Parse ground truth from OCRed Text
        ground_truth = parse_ocred_text(batch1_row['OCRed Text'])
        
        # Compare each domain
        for domain in domains:
            if domain not in ground_truth:
                continue
            
            gt_value = ground_truth[domain]
            ocr_value = output_row.get(domain)
            
            domain_result = results['domain_results'][domain]
            
            # Handle missing values
            if gt_value is None:
                domain_result['missing_in_ground_truth'] += 1
                domain_result['details'].append({
                    'filename': filename,
                    'status': 'missing_in_gt',
                    'ground_truth': gt_value,
                    'ocr_output': ocr_value
                })
                continue
            
            if pd.isna(ocr_value):
                domain_result['missing_in_output'] += 1
                domain_result['details'].append({
                    'filename': filename,
                    'status': 'missing_in_ocr',
                    'ground_truth': gt_value,
                    'ocr_output': None
                })
                continue
            
            # Determine field type and apply appropriate comparison
            is_numeric = domain in ['Net Worth', 'VAT', 'Gross Worth', 'Invoice Number']
            is_date = domain in ['Invoice Date']
            
            if is_date:
                # Extract date components and compare
                ocr_components = extract_date_components(ocr_value)
                gt_components = extract_date_components(gt_value)
                
                # For ambiguous dates (X/Y/YYYY where both X,Y <= 12), try both MM/DD and DD/MM interpretations
                is_match = False
                matched_gt_components = gt_components
                
                if ocr_components and gt_components:
                    if gt_components == ocr_components:
                        is_match = True
                    else:
                        # Try alternative interpretation for GT if it's ambiguous
                        if re.match(r'(\d{1,2})[/\-](\d{1,2})[/\-](\d{4})', str(gt_value)):
                            parts = re.match(r'(\d{1,2})[/\-](\d{1,2})[/\-](\d{4})', str(gt_value)).groups()
                            a, b, year = int(parts[0]), int(parts[1]), int(parts[2])
                            # Both <= 12: try swapped interpretation
                            if a <= 12 and b <= 12 and (a != b):
                                alt_gt_components = (year, b, a)  # Swap month and day
                                if alt_gt_components == ocr_components:
                                    is_match = True
                                    matched_gt_components = alt_gt_components
                
                score = 100 if is_match else 0
                
                domain_result['details'].append({
                    'filename': filename,
                    'status': 'match' if is_match else 'mismatch',
                    'ground_truth': gt_value,
                    'ground_truth_components': matched_gt_components,
                    'ocr_output': ocr_value,
                    'ocr_output_components': ocr_components,
                    'score': score
                })
            
            elif is_numeric:
                # Strict numeric comparison
                # gt_value = float(gt_value)
                is_match, diff = strict_compare(gt_value, ocr_value, tolerance=tolerance)
                
                
                domain_result['details'].append({
                    'filename': filename,
                    'status': 'match' if is_match else 'mismatch',
                    'ground_truth': gt_value,
                    'ocr_output': ocr_value,
                    'difference': diff,
                    'score': 100 if is_match else 0
                })
            
            else:
                # Fuzzy string comparison
                score, is_match = fuzzy_compare(str(gt_value), str(ocr_value), threshold=fuzzy_threshold)
                
                domain_result['details'].append({
                    'filename': filename,
                    'status': 'match' if is_match else 'mismatch',
                    'ground_truth': gt_value,
                    'ocr_output': ocr_value,
                    'similarity_score': score,
                    'score': 100 if is_match else score
                })
            
            # Update counters
            if is_match:
                domain_result['matches'] += 1
            else:
                domain_result['mismatches'] += 1
    
    # Calculate accuracy for each domain
    for domain in domains:
        domain_result = results['domain_results'][domain]
        total_compared = domain_result['matches'] + domain_result['mismatches']
        
        if total_compared > 0:
            domain_result['accuracy'] = (domain_result['matches'] / total_compared) * 100
    
    # Calculate overall accuracy
    all_matches = sum(r['matches'] for r in results['domain_results'].values())
    all_comparisons = sum(r['matches'] + r['mismatches'] for r in results['domain_results'].values())
    
    results['summary'] = {
        'total_domains': len(domains),
        'total_comparisons': all_comparisons,
        'total_matches': all_matches,
        'overall_accuracy': (all_matches / all_comparisons * 100) if all_comparisons > 0 else 0.0
    }
    
    return results


print("Validation function ready. Usage: validate_ocr_results(domains=['domain_name'])")

Validation function ready. Usage: validate_ocr_results(domains=['domain_name'])


In [151]:
# ============================================================================
# REPORTING AND VISUALIZATION
# ============================================================================

def print_validation_report(validation_results, show_mismatches=True, show_all_details=False):
    """
    Print a formatted validation report.
    
    Parameters:
    -----------
    validation_results : dict
        Output from validate_ocr_results()
    show_mismatches : bool
        If True, show details of all mismatches
    show_all_details : bool
        If True, show all details including matches
    """
    
    results = validation_results
    
    print("\n" + "="*80)
    print("OCR VALIDATION REPORT")
    print("="*80)
    
    print(f"\nFiles Validated: {results['files_matched']} / {results['total_files']}")
    if results['files_missing']:
        print(f"Files Missing from Ground Truth: {len(results['files_missing'])}")
    
    print("\n" + "-"*80)
    print("OVERALL ACCURACY")
    print("-"*80)
    print(f"Total Comparisons: {results['summary']['total_comparisons']}")
    print(f"Total Matches: {results['summary']['total_matches']}")
    print(f"Overall Accuracy: {results['summary']['overall_accuracy']:.2f}%")
    
    print("\n" + "-"*80)
    print("DOMAIN-BY-DOMAIN ACCURACY")
    print("-"*80)
    
    # Create summary table
    domain_data = []
    for domain in results['domains_validated']:
        dr = results['domain_results'][domain]
        total = dr['matches'] + dr['mismatches']
        domain_data.append({
            'Domain': domain,
            'Matches': dr['matches'],
            'Mismatches': dr['mismatches'],
            'Missing GT': dr['missing_in_ground_truth'],
            'Missing OCR': dr['missing_in_output'],
            'Accuracy %': f"{dr['accuracy']:.2f}%"
        })
    
    domain_df = pd.DataFrame(domain_data)
    print(domain_df.to_string(index=False))
    
    # Show details if requested
    if show_mismatches or show_all_details:
        print("\n" + "-"*80)
        print("DETAILED RESULTS")
        print("-"*80)
        
        for domain in results['domains_validated']:
            dr = results['domain_results'][domain]
            print(f"\n### {domain} ###")
            
            if show_all_details:
                # Show all details
                for detail in dr['details']:
                    print(f"  {detail['filename']}: {detail['status']}")
                    print(f"    GT:  {detail['ground_truth']}")
                    print(f"    OCR: {detail['ocr_output']}")
                    if 'similarity_score' in detail:
                        print(f"    Similarity: {detail['similarity_score']:.0f}%")
                    if 'difference' in detail and detail['difference'] is not None:
                        print(f"    Difference: {detail['difference']}")
                    print()
            else:
                # Show only mismatches
                mismatches = [d for d in dr['details'] if d['status'] == 'mismatch']
                if mismatches:
                    print(f"  {len(mismatches)} mismatches:")
                    for detail in mismatches[:10]:  # Show first 10
                        print(f"    {detail['filename']}")
                        print(f"      GT:  {detail['ground_truth']}")
                        print(f"      OCR: {detail['ocr_output']}")
                        if 'similarity_score' in detail:
                            print(f"      Similarity: {detail['similarity_score']:.0f}%")
                        if 'difference' in detail and detail['difference'] is not None:
                            print(f"      Difference: {detail['difference']}")
                    if len(mismatches) > 10:
                        print(f"    ... and {len(mismatches) - 10} more")
                else:
                    print(f"  ✓ All matches!")
    
    print("\n" + "="*80)


def get_mismatch_dataframe(validation_results, domain=None):
    """
    Extract mismatches as a pandas DataFrame for further analysis.
    
    Parameters:
    -----------
    validation_results : dict
        Output from validate_ocr_results()
    domain : str, optional
        Specific domain to extract. If None, returns all.
    
    Returns:
    --------
    df : pandas.DataFrame
        DataFrame with all mismatches
    """
    
    rows = []
    
    if domain:
        domains_to_check = [domain]
    else:
        domains_to_check = validation_results['domains_validated']
    
    for dom in domains_to_check:
        for detail in validation_results['domain_results'][dom]['details']:
            if detail['status'] == 'mismatch':
                row = {
                    'Domain': dom,
                    'Filename': detail['filename'],
                    'Ground Truth': detail['ground_truth'],
                    'OCR Output': detail['ocr_output'],
                    'Status': detail['status']
                }
                
                if 'similarity_score' in detail:
                    row['Similarity %'] = detail['similarity_score']
                if 'difference' in detail:
                    row['Difference'] = detail['difference']
                
                rows.append(row)
    
    return pd.DataFrame(rows)


print("Reporting functions ready.")
print("\nUsage:")
print("  results = validate_ocr_results(domains=['Seller Name', 'Invoice Date'])")
print("  print_validation_report(results)")
print("  mismatches_df = get_mismatch_dataframe(results)")

Reporting functions ready.

Usage:
  results = validate_ocr_results(domains=['Seller Name', 'Invoice Date'])
  print_validation_report(results)
  mismatches_df = get_mismatch_dataframe(results)


In [152]:
# Install required package
import subprocess
import sys

try:
    from fuzzywuzzy import fuzz
    print("✓ fuzzywuzzy already installed")
except ImportError:
    print("Installing fuzzywuzzy...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "fuzzywuzzy", "python-Levenshtein", "-q"])
    print("✓ fuzzywuzzy installed successfully")


✓ fuzzywuzzy already installed


In [153]:
# ============================================================================
# RUN FULL VALIDATION (ALL DOMAINS)
# ============================================================================

print("Starting full validation on all domains...")
print("This may take a moment...\n")

# Run validation on all domains
full_results = validate_ocr_results(output_df=output_df)

# Print report
print_validation_report(full_results, show_mismatches=True, show_all_details=False)


Starting full validation on all domains...
This may take a moment...

⚠️  Could not extract monetary values from Invoice Number: 48531206
⚠️  Could not extract monetary values from Invoice Number: 39571754
⚠️  Could not extract monetary values from Invoice Number: 60518602

OCR VALIDATION REPORT

Files Validated: 50 / 50

--------------------------------------------------------------------------------
OVERALL ACCURACY
--------------------------------------------------------------------------------
Total Comparisons: 441
Total Matches: 439
Overall Accuracy: 99.55%

--------------------------------------------------------------------------------
DOMAIN-BY-DOMAIN ACCURACY
--------------------------------------------------------------------------------
        Domain  Matches  Mismatches  Missing GT  Missing OCR Accuracy %
   Seller Name       50           0           0            0    100.00%
   Client Name       50           0           0            0    100.00%
 Seller Tax ID       50  

In [154]:
# ============================================================================
# VALIDATE SPECIFIC DOMAINS (CUSTOMIZE AS NEEDED)
# ============================================================================

FUZZY_MATCH_THRESHOLD = 90 
STRICT_NUMERIC_TOLERANCE = 0.1
# Example: Validate only specific domains
specific_domains = ['Seller Name', 'Invoice Number', 'Invoice Date']
specific_domains = ['Invoice Number']

print(f"Running validation on specific domains: {specific_domains}\n")
specific_results = validate_ocr_results(output_df=output_df,domains=specific_domains)

print_validation_report(specific_results, show_mismatches=True)


Running validation on specific domains: ['Invoice Number']

⚠️  Could not extract monetary values from Invoice Number: 48531206
⚠️  Could not extract monetary values from Invoice Number: 39571754
⚠️  Could not extract monetary values from Invoice Number: 60518602

OCR VALIDATION REPORT

Files Validated: 50 / 50

--------------------------------------------------------------------------------
OVERALL ACCURACY
--------------------------------------------------------------------------------
Total Comparisons: 50
Total Matches: 50
Overall Accuracy: 100.00%

--------------------------------------------------------------------------------
DOMAIN-BY-DOMAIN ACCURACY
--------------------------------------------------------------------------------
        Domain  Matches  Mismatches  Missing GT  Missing OCR Accuracy %
Invoice Number       50           0           0            0    100.00%

--------------------------------------------------------------------------------
DETAILED RESULTS
---------

In [155]:
# ============================================================================
# EXPORT AND ANALYZE MISMATCHES
# ============================================================================

# Get all mismatches as DataFrame
print("Exporting mismatches to DataFrame...\n")
all_mismatches_df = get_mismatch_dataframe(full_results)

print(f"Total mismatches: {len(all_mismatches_df)}\n")
print(all_mismatches_df.head(20))

# You can also get mismatches for a specific domain
# seller_name_mismatches = get_mismatch_dataframe(full_results, domain='Seller Name')
# print(f"\nSeller Name Mismatches: {len(seller_name_mismatches)}")
# print(seller_name_mismatches.head(10))

# Export to CSV if needed
# all_mismatches_df.to_csv(r'C:\Users\fkorniotis\code\makeathlon\makeathon-2026-WHOAMI-inform\.data\validation_mismatches.csv', index=False)
# print("\nMismatches exported to validation_mismatches.csv")


Exporting mismatches to DataFrame...

Total mismatches: 2

        Domain         Filename  Ground Truth  OCR Output    Status  \
0    Net Worth  batch1-0374.jpg           6.0         6.6  mismatch   
1  Gross Worth  batch1-0374.jpg           6.6         7.2  mismatch   

   Difference  
0         0.6  
1         0.6  


In [156]:
# ============================================================================
# SUMMARY OF RESULTS
# ============================================================================

print("\n" + "="*80)
print("VALIDATION SUMMARY WITH IMPROVED DATE LOGIC")
print("="*80)

print(f"\nOverall Accuracy: {full_results['summary']['overall_accuracy']:.2f}%")
print(f"Total Comparisons: {full_results['summary']['total_comparisons']}")
print(f"Total Matches: {full_results['summary']['total_matches']}")

print("\n" + "-"*80)
print("DOMAIN ACCURACIES")
print("-"*80)
for domain in full_results['domains_validated']:
    dr = full_results['domain_results'][domain]
    print(f"{domain:20s}: {dr['accuracy']:6.2f}% ({dr['matches']}/{dr['matches'] + dr['mismatches']})")

print("\n" + "-"*80)
print("DATE VALIDATION (with flexible format matching)")
print("-"*80)
date_results = full_results['domain_results']['Invoice Date']
print(f"Invoice Date Accuracy: {date_results['accuracy']:.2f}%")
print(f"Matches: {date_results['matches']}")
print(f"Mismatches: {date_results['mismatches']}")

if date_results['mismatches'] > 0:
    print(f"\nDate Mismatches ({date_results['mismatches']} total):")
    date_mismatches = [d for d in date_results['details'] if d['status'] == 'mismatch']
    for mismatch in date_mismatches[:5]:
        print(f"  {mismatch['filename']}")
        print(f"    GT:  {mismatch['ground_truth']} (parsed as {mismatch['ground_truth_components']})")
        print(f"    OCR: {mismatch['ocr_output']} (parsed as {mismatch['ocr_output_components']})")
else:
    print("\n✓ All Invoice Dates match perfectly!")

print("\n" + "="*80)



VALIDATION SUMMARY WITH IMPROVED DATE LOGIC

Overall Accuracy: 99.55%
Total Comparisons: 441
Total Matches: 439

--------------------------------------------------------------------------------
DOMAIN ACCURACIES
--------------------------------------------------------------------------------
Seller Name         : 100.00% (50/50)
Client Name         : 100.00% (50/50)
Seller Tax ID       : 100.00% (50/50)
Client Tax ID       : 100.00% (50/50)
Invoice Number      : 100.00% (50/50)
Invoice Date        : 100.00% (50/50)
Net Worth           :  97.87% (46/47)
VAT                 : 100.00% (47/47)
Gross Worth         :  97.87% (46/47)

--------------------------------------------------------------------------------
DATE VALIDATION (with flexible format matching)
--------------------------------------------------------------------------------
Invoice Date Accuracy: 100.00%
Matches: 50
Mismatches: 0

✓ All Invoice Dates match perfectly!



In [157]:
# Quick check of date matches
print("Date Matching Results:")
print(f"Invoice Date Accuracy: {full_results['domain_results']['Invoice Date']['accuracy']:.2f}%")
date_info = full_results['domain_results']['Invoice Date']
print(f"Matches: {date_info['matches']} | Mismatches: {date_info['mismatches']}")

# Show mismatches if any
mismatches = [d for d in date_info['details'] if d['status'] == 'mismatch']
if mismatches:
    print(f"\nRemaining Mismatches ({len(mismatches)}):")
    for m in mismatches:
        print(f"  {m['filename']}: GT={m['ground_truth']} vs OCR={m['ocr_output']}")
else:
    print("\n✓ Perfect Match - All dates now match!")


Date Matching Results:
Invoice Date Accuracy: 100.00%
Matches: 50 | Mismatches: 0

✓ Perfect Match - All dates now match!


In [158]:
# Final Overall Summary
print("\n" + "="*80)
print("FINAL VALIDATION RESULTS - ALL DOMAINS")
print("="*80)

print(f"\nFiles Validated: {full_results['files_matched']}/{full_results['total_files']}")
print(f"Overall Accuracy: {full_results['summary']['overall_accuracy']:.2f}%")
print(f"Total Matches: {full_results['summary']['total_matches']}/{full_results['summary']['total_comparisons']}")

print("\n" + "-"*80)
print("DOMAIN-BY-DOMAIN BREAKDOWN")
print("-"*80)

for domain in full_results['domains_validated']:
    dr = full_results['domain_results'][domain]
    total = dr['matches'] + dr['mismatches']
    
    if total == 0:
        acc_str = "N/A"
    else:
        acc_str = f"{dr['accuracy']:.2f}%"
    
    print(f"{domain:20s}: {acc_str:>8s} | Matches: {dr['matches']:>2d} | Mismatches: {dr['mismatches']:>2d}")

print("\n" + "-"*80)
print("KEY FINDINGS")
print("-"*80)

# Count domains with perfect accuracy
perfect_domains = [d for d in full_results['domains_validated'] 
                   if full_results['domain_results'][d]['accuracy'] == 100.0 
                   and (full_results['domain_results'][d]['matches'] + full_results['domain_results'][d]['mismatches']) > 0]

print(f"✓ Perfect Matches ({len(perfect_domains)} domains):")
for d in perfect_domains:
    print(f"  - {d}")

print("\n" + "="*80)



FINAL VALIDATION RESULTS - ALL DOMAINS

Files Validated: 50/50
Overall Accuracy: 99.55%
Total Matches: 439/441

--------------------------------------------------------------------------------
DOMAIN-BY-DOMAIN BREAKDOWN
--------------------------------------------------------------------------------
Seller Name         :  100.00% | Matches: 50 | Mismatches:  0
Client Name         :  100.00% | Matches: 50 | Mismatches:  0
Seller Tax ID       :  100.00% | Matches: 50 | Mismatches:  0
Client Tax ID       :  100.00% | Matches: 50 | Mismatches:  0
Invoice Number      :  100.00% | Matches: 50 | Mismatches:  0
Invoice Date        :  100.00% | Matches: 50 | Mismatches:  0
Net Worth           :   97.87% | Matches: 46 | Mismatches:  1
VAT                 :  100.00% | Matches: 47 | Mismatches:  0
Gross Worth         :   97.87% | Matches: 46 | Mismatches:  1

--------------------------------------------------------------------------------
KEY FINDINGS
---------------------------------------------

In [159]:
# # Investigate Invoice Number extraction issues
# print("Invoice Number Analysis:")
# print("-" * 80)

# inv_num_details = full_results['domain_results']['Invoice Number']['details']

# # Count different status types
# missing_gt = [d for d in inv_num_details if d['status'] == 'missing_in_gt']
# missing_ocr = [d for d in inv_num_details if d['status'] == 'missing_in_ocr']
# mismatches = [d for d in inv_num_details if d['status'] == 'mismatch']

# print(f"Missing in Ground Truth: {len(missing_gt)}")
# print(f"Missing in OCR Output: {len(missing_ocr)}")
# print(f"Mismatches: {len(mismatches)}")

# if missing_gt:
#     print(f"\nExamples of files where Invoice Number couldn't be extracted from ground truth:")
#     for detail in missing_gt[:3]:
#         print(f"  - {detail['filename']}")
        
# if mismatches:
#     print(f"\nSample Mismatches (first 5):")
#     for detail in mismatches[:5]:
#         print(f"  {detail['filename']}: GT={detail['ground_truth']} | OCR={detail['ocr_output']}")

# # Check one OCRed text manually to see the format
# if len(missing_gt) > 0:
#     sample_file = missing_gt[0]['filename']
#     idx = batch1_index.get(sample_file)
#     if idx is not None:
#         print(f"\nSample OCRed text (from {sample_file}):")
#         sample_text = batch1_df.iloc[idx]['OCRed Text']
#         # Find the Invoice line
#         for line in sample_text.split('\n')[:20]:
#             if 'invoice' in line.lower():
#                 print(f"  {line}")


In [160]:
# # Debug Invoice Number - check actual vs expected
# print("Invoice Number Issue Investigation:")
# print("-" * 80)

# # Check sample
# filename = 'batch1-0331.jpg'

# # Find in output_df
# ocr_row = output_df[output_df['filename'] == filename]
# ocr_inv = ocr_row['Invoice Number'].values[0]

# # Find in batch1_df
# batch1_row = batch1_df[batch1_df['File Name'].str.contains(filename, na=False)]
# gt_text = parse_ocred_text(batch1_row['OCRed Text'].values[0])
# gt_inv = gt_text['Invoice Number']

# print(f"File: {filename}")
# print(f"GT Invoice Number: {gt_inv!r} (type: {type(gt_inv).__name__})")
# print(f"OCR Invoice Number: {ocr_inv!r} (type: {type(ocr_inv).__name__})")
# print(f"\nDirect comparison (==): {gt_inv == ocr_inv}")

# # Check if it's a string vs int issue
# if isinstance(gt_inv, str) and isinstance(ocr_inv, int):
#     print(f"String vs Int detected")
#     print(f"GT as int: {int(gt_inv)}")
#     print(f"Are they equal as ints? {int(gt_inv) == ocr_inv}")
